In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

In [2]:
df = pd.read_csv('../data/Telco-Customer-Churn.csv')
print("데이터 로드 완료. 원본 데이터 크기:", df.shape)
df.head(3)

데이터 로드 완료. 원본 데이터 크기: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [3]:
# ==========================================
# 기본 전처리 및 결측치 처리
# ==========================================
# 불필요한 고유 식별자 피처 제거
df = df.drop('customerID', axis=1)


# TotalCharges 열의 공백 결측치를 0으로 변환 후 숫자형으로 캐스팅
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [4]:
# ==========================================
# 핵심 피처 생성 (Feature Engineering)
# ==========================================

# 부가 서비스 관여도 (Service Engagement Score)
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']

df['Service_Engagement'] = 0
for service in services:
    df['Service_Engagement'] += (df[service] == 'Yes').astype(int)  # 6가지 중 1개라도 Yes 라면 Service_Engagement에 1로 들어감
#df["Service_Engagement"].describe()


# 가입 기간 범주화 (Tenure Group)
def tenure_to_group(tenure):
    if tenure <= 12:
        return 'New(0-1yr)'
    elif tenure <= 36:
        return 'Standard(1-3yr)'
    else:
        return 'Loyal(3yr+)'
df['Tenure_Group'] = df['tenure'].apply(tenure_to_group)

df.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Service_Engagement,Tenure_Group
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1,New(0-1yr)
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,One year,No,Mailed check,56.95,1889.50,No,2,Standard(1-3yr)
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,2,New(0-1yr)
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,3,Loyal(3yr+)
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0,New(0-1yr)


In [5]:
# ==========================================
# 범주형 데이터 인코딩
# ==========================================
# 이진형 데이터 변환 (Label Encoding)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# 다중 클래스 범주 데이터 변환 (One-Hot Encoding)
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
              'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
              'Contract', 'PaymentMethod', 'Tenure_Group']

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

df.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Tenure_Group_New(0-1yr),Tenure_Group_Standard(1-3yr)
0,0,0,1,0,1,0,1,29.85,29.85,0,...,False,False,False,False,False,False,True,False,True,False
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,False,False,False,True,False,False,False,True,False,True
2,1,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,False,False,False,False,True,True,False
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,False,False,False,True,False,False,False,False,False,False
4,0,0,0,0,2,1,1,70.70,151.65,1,...,False,False,False,False,False,False,True,False,True,False


# 상관관계 분석

In [6]:
# ==========================================
# 상관관계 분석
# ==========================================
corr_with_churn = df.corr()['Churn'].sort_values(ascending=False)
print("\n=== 고객 이탈(Churn)과 강한 양의 상관관계를 가진 피처 Top 5 ===")
print(corr_with_churn.drop('Churn').head(5))

print("\n=== 고객 이탈(Churn)과 강한 음의 상관관계를 가진 피처 Top 5 ===")
print(corr_with_churn.drop('Churn').tail(5))



=== 고객 이탈(Churn)과 강한 양의 상관관계를 가진 피처 Top 5 ===
Tenure_Group_New(0-1yr)           0.317580
InternetService_Fiber optic       0.308020
PaymentMethod_Electronic check    0.301919
MonthlyCharges                    0.193356
PaperlessBilling                  0.191825
Name: Churn, dtype: float64

=== 고객 이탈(Churn)과 강한 음의 상관관계를 가진 피처 Top 5 ===
TechSupport_No internet service        -0.227890
DeviceProtection_No internet service   -0.227890
StreamingTV_No internet service        -0.227890
Contract_Two year                      -0.302253
tenure                                 -0.352229
Name: Churn, dtype: float64


# 모델별 성능, feature importance 평가

# 모델1 - Random Forest

In [7]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 학습/테스트 셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 예측 및 성능 평가
y_pred = rf_model.predict(X_test)
print("\n=== 예측 모델 성능 (Classification Report) ===")
print(classification_report(y_test, y_pred))

# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(5):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")




=== 예측 모델 성능 (Classification Report) ===
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1036
           1       0.66      0.48      0.56       373

    accuracy                           0.80      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.80      0.79      1409


=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. TotalCharges (0.1720)
2. MonthlyCharges (0.1593)
3. tenure (0.1446)
4. Service_Engagement (0.0406)
5. Tenure_Group_New(0-1yr) (0.0387)


In [8]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가 - class_weight = 'balanced' 추가
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 학습/테스트 셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# 예측 및 성능 평가
y_pred = rf_model.predict(X_test)
print("\n=== 예측 모델 성능 (Classification Report) ===")
print(classification_report(y_test, y_pred))

# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(7):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")




=== 예측 모델 성능 (Classification Report) ===
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1036
           1       0.65      0.45      0.53       373

    accuracy                           0.79      1409
   macro avg       0.74      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409


=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. TotalCharges (0.1551)
2. tenure (0.1506)
3. MonthlyCharges (0.1449)
4. Contract_Two year (0.0469)
5. InternetService_Fiber optic (0.0421)
6. Service_Engagement (0.0362)
7. Tenure_Group_New(0-1yr) (0.0350)


class_weight로 균형을 맞추고 나니 중요 피처에서 Service Engagement와 Tenure_Group_new 가 밀리고 상관관계 분석에서 봤던 Contract_Two year (0.0469) & InternetService_Fiber optic (0.0421) 가 올라옴. 
=> 이탈에 service Engagement가 엄청 큰 영향을 주진 않는 듯? 그러나 그래도 모델 학습에 유의미한 도움은 준다.

In [9]:
#가입 기간 대비 평균 납부액 특성 추가
df['Avg_Monthly_Spent'] = df['TotalCharges'] / (df['tenure'] + 1) # 0으로 나누기 방지 (+1)

In [10]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가 - class_weight balanced + Avg_Monthly_Spent 특성 추가 + 검증셋으로 threshold 조정
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 1차: train / test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2차: train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25,  # 0.25 * 0.8 = 0.2
    stratify=y_train_full,
    random_state=42
)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)


# 과적합 검사
y_train_pred = rf_model.predict(X_train)
y_val_pred = rf_model.predict(X_val)

print("Train F1:", f1_score(y_train, y_train_pred))
print("Val F1:", f1_score(y_val, y_val_pred))



# 최적 임계값 선정
y_val_proba = rf_model.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
best_t = 0
best_f1 = 0
for t in thresholds:
    y_pred = (y_val_proba >= t).astype(int)
    score = f1_score(y_val, y_pred)
    
    if score > best_f1:
        best_f1 = score
        best_t = t
print("\nBest threshold:", best_t)
print("Best F1:", best_f1)

y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
y_pred_custom = (y_pred_proba >= best_t).astype(int)


# 성능 지표 출력
print(f"=== [임계값 {best_t:.2f} 적용] 성능 보고서 ===")
print(classification_report(y_test, y_pred_custom))

# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(5):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")



Train F1: 0.9964349376114082
Val F1: 0.5394321766561514

Best threshold: 0.30000000000000004
Best F1: 0.6122905027932961
=== [임계값 0.30 적용] 성능 보고서 ===
              precision    recall  f1-score   support

           0       0.88      0.77      0.82      1035
           1       0.53      0.72      0.61       374

    accuracy                           0.76      1409
   macro avg       0.71      0.74      0.72      1409
weighted avg       0.79      0.76      0.77      1409


=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. TotalCharges (0.1274)
2. tenure (0.1255)
3. MonthlyCharges (0.1174)
4. Avg_Monthly_Spent (0.1145)
5. Contract_Two year (0.0631)


4위에 올라옴. 유의미한 피처가 될 수 있음을 확인!

# 모델 변경 - XGBClassifier

In [11]:
from xgboost import XGBClassifier

In [12]:
# 데이터 분리 
X = df.drop('Churn', axis=1)
y = df['Churn']

# 1차: train / test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2차: train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25,  # 0.25 * 0.8 = 0.2
    stratify=y_train_full,
    random_state=42
)

# XGBoost 
# 불균형 가중치(scale_pos_weight) = (음성 샘플 수 / 양성 샘플 수) 대략 3 정도로 설정
model_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,    # 5 -> 3,4 과적합 방지
    scale_pos_weight=4,  # 이탈 데이터(1)에 약 3배의 가중치 부여
    #min_child_weight=30,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model_xgb.fit(X_train, y_train)


# 과적합 검사
y_train_pred = model_xgb.predict(X_train)
y_val_pred = model_xgb.predict(X_val)

print("Train F1:", f1_score(y_train, y_train_pred))
print("Val F1:", f1_score(y_val, y_val_pred))



# 최적 임계값 선정
y_val_proba = model_xgb.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
best_t = 0
best_f1 = 0
for t in thresholds:
    y_pred = (y_val_proba >= t).astype(int)
    score = f1_score(y_val, y_pred)
    
    if score > best_f1:
        best_f1 = score
        best_t = t
print("\nBest threshold:", best_t)
print("Best F1:", best_f1)

y_pred_proba = model_xgb.predict_proba(X_test)[:, 1]
y_pred_custom = (y_pred_proba >= best_t).astype(int)


# 성능 지표 출력
print(f"=== [임계값 {best_t:.2f} 적용] 성능 보고서 ===")
print(classification_report(y_test, y_pred_custom))

Train F1: 0.6487523992322457
Val F1: 0.6063522617901829

Best threshold: 0.7000000000000002
Best F1: 0.6326034063260341
=== [임계값 0.70 적용] 성능 보고서 ===
              precision    recall  f1-score   support

           0       0.88      0.80      0.84      1035
           1       0.56      0.68      0.62       374

    accuracy                           0.77      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.77      0.78      1409



c:\Users\EZ\module_gaia\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## 모델 확률값으로 데이터 분석1 - 이탈 확률로 나누기

In [13]:
# 분석용 데이터프레임 생성
analysis_df = X_test.copy()

# "이탈 확률" 받아와 기존 테스트 데이터에 컬럼 추가 (X_test에 인덱스가 유지되어 있다고 가정)
analysis_df['Churn_Prob'] = y_pred_proba  # XGBoost 모델이 계산한 확률값 (0~1)
analysis_df['Actual_Churn'] = y_test # 실제 이탈 여부 (비교용)

# 2. 확률 구간(Grade)으로 그룹 나누기 (0~100%를 5개 구간으로 구분)
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
analysis_df['Prob_Grade'] = pd.cut(analysis_df['Churn_Prob'], bins=bins, labels=labels)

# 3. 그룹별 기초 통계 지표 추출
# 분석하고 싶은 주요 컬럼들을 선정합니다.
metrics = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Avg_Monthly_Spent', 'Actual_Churn']

# 그룹별 평균값 계산
group_stats = analysis_df.groupby('Prob_Grade')[metrics].mean()

# 그룹별 고객 수(Count) 추가
group_stats['Customer_Count'] = analysis_df.groupby('Prob_Grade').size()

print("=== [이탈 확률 구간별 기초 통계 분석] ===")
display(group_stats)



# 4. 특정 고위험군(Very High) 고객 리스트만 따로 뽑기
high_risk_customers = analysis_df[analysis_df['Prob_Grade'] == 'Very High'].sort_values(by='Churn_Prob', ascending=False)

print(f"\n=== [초고위험군(80% 이상) 샘플 5건] ===")
display(high_risk_customers.head(5))

=== [이탈 확률 구간별 기초 통계 분석] ===


,tenure,MonthlyCharges,TotalCharges,Avg_Monthly_Spent,Actual_Churn,Customer_Count
Prob_Grade,,,,,,
Very Low,50.517544,49.128289,2810.406579,47.902218,0.032895,456
Low,36.735000,60.760000,2728.850500,58.349447,0.130000,200
Medium,30.039548,71.454237,2591.661017,67.565931,0.225989,177
High,22.460265,70.682450,1945.360099,64.120549,0.377483,302
Very High,9.072993,79.391241,832.769526,62.278653,0.653285,274



=== [초고위험군(80% 이상) 샘플 5건] ===


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Service_Engagement,...,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Tenure_Group_New(0-1yr),Tenure_Group_Standard(1-3yr),Avg_Monthly_Spent,Churn_Prob,Actual_Churn,Prob_Grade
6623,1,1,0,0,1,1,1,76.45,76.45,0,...,False,False,True,False,True,False,38.225,0.976881,1,Very High
2464,0,0,0,0,1,1,1,77.15,77.15,0,...,False,False,True,False,True,False,38.575,0.972998,1,Very High
4585,0,1,0,0,1,1,1,85.05,85.05,1,...,False,False,True,False,True,False,42.525,0.972905,1,Very High
3380,1,1,1,0,1,1,1,95.10,95.10,2,...,False,False,True,False,True,False,47.550,0.972852,1,Very High
6866,1,0,0,0,1,1,1,95.45,95.45,2,...,False,False,True,False,True,False,47.725,0.972189,1,Very High


## 모델 확률값으로 데이터 분석2 - 마케팅 액션 등급별로 나누기

In [14]:
# 분석용 데이터프레임 복사 및 확률 추가
action_df = X_test.copy()
action_df['Churn_Prob'] = y_pred_proba  # 모델의 예측 확률
action_df['Actual_Churn'] = y_test.values

# 마케팅 액션 등급(Action_Grade) 생성
def assign_action_grade(row):
    prob = row['Churn_Prob']
    tenure = row['tenure']
    monthly_charges = row['MonthlyCharges']

    if prob >= 0.7:
        if monthly_charges > action_df['MonthlyCharges'].median():
            return '1_Emergency_VIP'  # 고액 결제 중인 초고위험군 (최우선 방어)
        return '2_High_Risk'           # 일반 초고위험군
    
    elif 0.4 <= prob < 0.7:
        if tenure > 24:
            return '3_Loyal_At_Risk'    # 장기 이용자 중 이탈 징후 (혜택 강화)
        return '4_Potential_Churn'      # 잠재적 이탈군
    
    else:
        return '5_Safe_Zone'            # 안정권

action_df['Action_Grade'] = action_df.apply(assign_action_grade, axis=1)

# 3. 등급별 대응 전략(Action_Strategy) 매핑
strategy_map = {
    '1_Emergency_VIP': '전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문)',
    '2_High_Risk': '단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사',
    '3_Loyal_At_Risk': '장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안',
    '4_Potential_Churn': '부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도',
    '5_Safe_Zone': '현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토'
}

action_df['Action_Strategy'] = action_df['Action_Grade'].map(strategy_map)

# 4. 결과 요약 통계 확인
action_summary = action_df.groupby('Action_Grade').agg({
    'Churn_Prob': 'mean',
    'MonthlyCharges': 'mean',
    'tenure': 'mean',
    'Action_Strategy': 'first',
    'Actual_Churn': 'count'
}).rename(columns={'Actual_Churn': 'Customer_Count'})

print("=== [고객 등급별 마케팅 액션 요약] ===")
display(action_summary.sort_index())

# 5. 실무용: 즉시 연락이 필요한 'Emergency_VIP' 명단 추출
emergency_list = action_df[action_df['Action_Grade'] == '1_Emergency_VIP'].sort_values(by='Churn_Prob', ascending=False)
print(f"\n[알림] 최우선 관리 대상(Emergency_VIP) {len(emergency_list)}명을 발견했습니다.")

=== [고객 등급별 마케팅 액션 요약] ===


,Churn_Prob,MonthlyCharges,tenure,Action_Strategy,Customer_Count
Action_Grade,,,,,
1_Emergency_VIP,0.836357,86.831858,16.713864,전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문),339
2_High_Risk,0.801178,43.584874,3.621849,단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사,119
3_Loyal_At_Risk,0.554039,91.292123,47.465753,장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안,146
4_Potential_Churn,0.573591,52.318456,10.463087,부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도,149
5_Safe_Zone,0.144003,52.674543,46.315549,현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토,656



[알림] 최우선 관리 대상(Emergency_VIP) 339명을 발견했습니다.


# 모델변경 - GradientBoostingClassifier

In [15]:
from sklearn.ensemble import GradientBoostingClassifier

X = df.drop('Churn', axis=1)
y = df['Churn']

# 1차: train / test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2차: train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25,  # 0.25 * 0.8 = 0.2
    stratify=y_train_full,
    random_state=42
)


model_gbr = GradientBoostingClassifier(random_state=42)
model_gbr.fit(X_train, y_train)

# 과적합 검사
y_train_pred = model_gbr.predict(X_train)
y_val_pred = model_gbr.predict(X_val)

print("Train F1:", f1_score(y_train, y_train_pred))
print("Val F1:", f1_score(y_val, y_val_pred))

# 임계값 조정
y_val_proba = model_gbr.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
best_t = 0
best_f1 = 0
for t in thresholds:
    y_pred = (y_val_proba >= t).astype(int)
    score = f1_score(y_val, y_pred)
    
    if score > best_f1:
        best_f1 = score
        best_t = t
print("\nBest threshold:", best_t)
print("Best F1:", best_f1)

y_pred_proba = model_gbr.predict_proba(X_test)[:, 1]
y_pred_custom = (y_pred_proba >= best_t).astype(int)

# 성능 지표 출력
print(f"\n=== [임계값 {best_t} 적용] 성능 보고서 ===")
print(classification_report(y_test, y_pred_custom))


Train F1: 0.6438631790744467
Val F1: 0.5727699530516432

Best threshold: 0.40000000000000013
Best F1: 0.6321381142098274

=== [임계값 0.40000000000000013 적용] 성능 보고서 ===
              precision    recall  f1-score   support

           0       0.87      0.85      0.86      1035
           1       0.60      0.64      0.62       374

    accuracy                           0.79      1409
   macro avg       0.74      0.75      0.74      1409
weighted avg       0.80      0.79      0.80      1409



## 모델 확률값으로 데이터 분석 - 이탈 확률로 나누기

In [16]:
# 분석용 데이터프레임 생성
analysis_df = X_test.copy()

# "이탈 확률" 받아와 기존 테스트 데이터에 컬럼 추가 (X_test에 인덱스가 유지되어 있다고 가정)
analysis_df['Churn_Prob'] = y_pred_proba  # XGBoost 모델이 계산한 확률값 (0~1)
analysis_df['Actual_Churn'] = y_test # 실제 이탈 여부 (비교용)

# 2. 확률 구간(Grade)으로 그룹 나누기 (0~100%를 5개 구간으로 구분)
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
analysis_df['Prob_Grade'] = pd.cut(analysis_df['Churn_Prob'], bins=bins, labels=labels)

# 3. 그룹별 기초 통계 지표 추출
# 분석하고 싶은 주요 컬럼들을 선정합니다.
metrics = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Avg_Monthly_Spent', 'Actual_Churn']

# 그룹별 평균값 계산
group_stats = analysis_df.groupby('Prob_Grade')[metrics].mean()

# 그룹별 고객 수(Count) 추가
group_stats['Customer_Count'] = analysis_df.groupby('Prob_Grade').size()

print("=== [이탈 확률 구간별 기초 통계 분석] ===")
display(group_stats)



# 4. 특정 고위험군(Very High) 고객 리스트만 따로 뽑기
high_risk_customers = analysis_df[analysis_df['Prob_Grade'] == 'Very High'].sort_values(by='Churn_Prob', ascending=False)

print(f"\n=== [초고위험군(80% 이상) 샘플 5건] ===")
display(high_risk_customers.head(5))

=== [이탈 확률 구간별 기초 통계 분석] ===


,tenure,MonthlyCharges,TotalCharges,Avg_Monthly_Spent,Actual_Churn,Customer_Count
Prob_Grade,,,,,,
Very Low,44.459677,55.219019,2793.603024,53.432654,0.077957,744
Low,26.522388,68.672575,2214.932276,63.393242,0.283582,268
Medium,17.649289,77.187441,1592.255687,68.037237,0.492891,211
High,7.165354,76.397244,630.857480,61.052463,0.716535,127
Very High,2.525424,81.779661,230.334746,51.668203,0.762712,59



=== [초고위험군(80% 이상) 샘플 5건] ===


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Service_Engagement,...,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Tenure_Group_New(0-1yr),Tenure_Group_Standard(1-3yr),Avg_Monthly_Spent,Churn_Prob,Actual_Churn,Prob_Grade
6623,1,1,0,0,1,1,1,76.45,76.45,0,...,False,False,True,False,True,False,38.225,0.931294,1,Very High
4585,0,1,0,0,1,1,1,85.05,85.05,1,...,False,False,True,False,True,False,42.525,0.928731,1,Very High
3380,1,1,1,0,1,1,1,95.10,95.10,2,...,False,False,True,False,True,False,47.550,0.928314,1,Very High
2194,1,0,0,0,1,1,1,79.50,79.50,1,...,False,False,True,False,True,False,39.750,0.924167,1,Very High
6866,1,0,0,0,1,1,1,95.45,95.45,2,...,False,False,True,False,True,False,47.725,0.920795,1,Very High


## 모델 확률값으로 데이터 분석2 - 마케팅 액션 등급별로 나누기

In [17]:
# 분석용 데이터프레임 복사 및 확률 추가
action_df = X_test.copy()
action_df['Churn_Prob'] = y_pred_proba  # 모델의 예측 확률
action_df['Actual_Churn'] = y_test.values

# 마케팅 액션 등급(Action_Grade) 생성
def assign_action_grade(row):
    prob = row['Churn_Prob']
    tenure = row['tenure']
    monthly_charges = row['MonthlyCharges']

    if prob >= 0.7:
        if monthly_charges > action_df['MonthlyCharges'].median():
            return '1_Emergency_VIP'  # 고액 결제 중인 초고위험군 (최우선 방어)
        return '2_High_Risk'           # 일반 초고위험군
    
    elif 0.4 <= prob < 0.7:
        if tenure > 24:
            return '3_Loyal_At_Risk'    # 장기 이용자 중 이탈 징후 (혜택 강화)
        return '4_Potential_Churn'      # 잠재적 이탈군
    
    else:
        return '5_Safe_Zone'            # 안정권

action_df['Action_Grade'] = action_df.apply(assign_action_grade, axis=1)

# 3. 등급별 대응 전략(Action_Strategy) 매핑
strategy_map = {
    '1_Emergency_VIP': '전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문)',
    '2_High_Risk': '단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사',
    '3_Loyal_At_Risk': '장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안',
    '4_Potential_Churn': '부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도',
    '5_Safe_Zone': '현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토'
}

action_df['Action_Strategy'] = action_df['Action_Grade'].map(strategy_map)

# 4. 결과 요약 통계 확인
action_summary = action_df.groupby('Action_Grade').agg({
    'Churn_Prob': 'mean',
    'MonthlyCharges': 'mean',
    'tenure': 'mean',
    'Action_Strategy': 'first',
    'Actual_Churn': 'count'
}).rename(columns={'Actual_Churn': 'Customer_Count'})

print("=== [고객 등급별 마케팅 액션 요약] ===")
display(action_summary.sort_index())

# 5. 실무용: 즉시 연락이 필요한 'Emergency_VIP' 명단 추출
emergency_list = action_df[action_df['Action_Grade'] == '1_Emergency_VIP'].sort_values(by='Churn_Prob', ascending=False)
print(f"\n[알림] 최우선 관리 대상(Emergency_VIP) {len(emergency_list)}명을 발견했습니다.")

=== [고객 등급별 마케팅 액션 요약] ===


,Churn_Prob,MonthlyCharges,tenure,Action_Strategy,Customer_Count
Action_Grade,,,,,
1_Emergency_VIP,0.801788,84.263684,4.084211,전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문),95
2_High_Risk,0.793221,46.969565,1.434783,단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사,23
3_Loyal_At_Risk,0.482656,95.027273,37.378788,장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안,66
4_Potential_Churn,0.543813,72.567371,8.896714,부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도,213
5_Safe_Zone,0.130516,58.781818,39.709486,현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토,1012



[알림] 최우선 관리 대상(Emergency_VIP) 95명을 발견했습니다.
